## Config

In [ ]:

"""
Configuration initiale et imports
Exécuter cette cellule en premier
"""
%matplotlib inline
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import matplotlib
matplotlib.use('Agg')  # Changer en 'inline' pour Jupyter

import spacy
from lxml import etree
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Configuration des chemins
OUTPUT_DIR = "output_lemmes"
VISUALIZATIONS_DIR = "humanstica_output"
TEI_NS = "http://www.tei-c.org/ns/1.0"
NSMAP = {'tei': TEI_NS}
NON_TRAITE = "NonTraite.txt"
PEINTURE = "peinture.txt"

# Créer les dossiers de sortie
Path(VISUALIZATIONS_DIR).mkdir(exist_ok=True)

print("✓ Configuration chargée")


✓ Configuration chargée


### Stopwords

In [19]:
"""
Charger les modèles spaCy et les stopwords
"""

try:
    nlp_fr = spacy.load('fr_core_news_sm')
    nlp_it = spacy.load('it_core_news_sm')
    
    more_stopwords_fra = ['ay','cy','dict','-','pl','a.','b.','c.','d.','e.','f.','g.','h.','i.','j.','k.','l.','m.','n.','o.','p.','q.','r.','s.','t.','u.','v.','w.','x.','y.','z.','-là','apre','chose', 'faict','ester','aller','autant','celer','trop','beaucoup','estr','étoit','temp','rien','avoit', 'aucun','tre', 'ilz', 'aprer', 'estoier', 'faire', 'estre', 'fut', 'bien', 'estoit', 'mesme', 'luy', 'faut', 'faisoit', 'fit', 'costé', 'non', '\'', 'l', 'd', 'qu', 'j', 'n', 'c', 's']
    more_stopwords_ita = ['gl','quell','a.','b.','c.','d.','e.','f.','g.','h.','i.','j.','k.','l.','m.','n.','o.','p.','q.','r.','s.','t.','u.','v.','w.','x.','y.','z.','ch','cose', 'altra', 'como','esso','nostr','stare','lo','si','lor','andare','qual', 'altre', 'ciò', 'da', 'il', 'da il', 'di il', 'il di', 'su il', 'con il', 'lo', 'et', 'l', '\'', 's.', 'in il', 'o', 'de', 'd', 'a il']
    
    # Ajouter les stopwords personnalisés
    for item in more_stopwords_fra:
        nlp_fr.Defaults.stop_words.add(item)
    
    for item in more_stopwords_ita:
        nlp_it.Defaults.stop_words.add(item)
    
    # Récupérer les ensembles de stopwords
    STOPWORDS_FR = nlp_fr.Defaults.stop_words
    STOPWORDS_IT = nlp_it.Defaults.stop_words
    
    print("✓ Stopwords chargés depuis spaCy")
    print(f"  - Français: {len(STOPWORDS_FR)} stopwords")
    print(f"  - Italien: {len(STOPWORDS_IT)} stopwords")
    
except Exception as e:
    print(f"⚠ Impossible de charger les modèles spaCy: {e}")
    print("  Utilisation des stopwords par défaut")
    STOPWORDS_FR = {'le', 'la', 'les', 'un', 'une', 'des', 'de', 'du', 'et', 'ou'}
    STOPWORDS_IT = {'il', 'lo', 'la', 'i', 'gli', 'le', 'un', 'uno', 'una', 'di'}

✓ Stopwords chargés depuis spaCy
  - Français: 578 stopwords
  - Italien: 677 stopwords


### Fonctions chargement du corpus

In [4]:

"""
Fonctions de base pour l'extraction et le filtrage
"""

def should_exclude_by_pos(pos):
    """Vérifie si un POS doit être exclu (ponctuation, nombres)."""
    excluded_pos = {'PUNCT', 'NUM', 'SYM', 'X'}
    return pos in excluded_pos

def filter_dataframe(df):
    """Filtre le dataframe pour enlever ponctuation et nombres selon POS."""
    return df[~df['pos'].apply(should_exclude_by_pos)]

def extract_tokens_from_xml(xml_path):
    """Extrait tous les tokens annotés d'un fichier XML enrichi."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()
    
    tokens_data = []
    w_elements = root.xpath('.//tei:w | .//w', namespaces=NSMAP)
    
    for w in w_elements:
        token_text = w.text if w.text else ''
        lemma = w.get('lemma', '')
        pos = w.get('pos', '')
        
        if token_text and lemma:
            tokens_data.append({
                'token': token_text.lower(),
                'lemma': lemma.lower(),
                'pos': pos
            })
    
    return tokens_data

def extract_named_entities_from_xml(xml_path):
    """Extrait les entités nommées (persName et placeName) d'un fichier XML."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()
    
    entities = []
    
    # Extraire les persName
    pers_elements = root.xpath('.//tei:persName | .//persName', namespaces=NSMAP)
    for pers in pers_elements:
        text = ''.join(pers.itertext()).strip()
        ref = pers.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'personne'
            })
    
    # Extraire les placeName
    place_elements = root.xpath('.//tei:placeName | .//placeName', namespaces=NSMAP)
    for place in place_elements:
        text = ''.join(place.itertext()).strip()
        ref = place.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'lieu'
            })
    
    # Extraire les placeName
    object_elements = root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP)
    for object in object_elements:
        text = ''.join(object.itertext()).strip()
        ref = object.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'object'
            })
    
    return entities

def calculate_statistics(df, language):
    """Calcule les statistiques descriptives du corpus."""
    stats = {
        'langue': language,
        'nb_tokens': len(df),
        'nb_tokens_uniques': df['token'].nunique(),
        'nb_lemmes_uniques': df['lemma'].nunique(),
        'ratio_diversite': df['lemma'].nunique() / len(df) if len(df) > 0 else 0,
        'longueur_moyenne_token': df['token'].str.len().mean()
    }
    return stats

def extract_ngrams(df, n=2):
    """Extrait les n-grammes d'un corpus."""
    lemmas = df['lemma'].values
    ngrams = []
    
    for i in range(len(lemmas) - n + 1):
        ngram = tuple(lemmas[i:i+n])
        ngrams.append(ngram)
    
    return Counter(ngrams)

print("✓ Fonctions utilitaires chargées")



✓ Fonctions utilitaires chargées


In [5]:
"""
CHARGEMENT DES DONNÉES
Charger les corpus avec filtrage par sous-corpus et liste NON_TRAITE.
Les noms de fichiers commencent par le nom du sous-corpus en MAJUSCULE.
Ex: PEINTURE_Auteur_Nom.xml
"""

def load_text_list(filepath, list_type="liste"):
    """Charge une liste de noms depuis un fichier texte."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            items = [line.strip() for line in f if line.strip()]
        print(f"✓ {list_type.capitalize()} chargée : {len(items)} entrées")
        return items
    except FileNotFoundError:
        print(f"⚠ Fichier {filepath} non trouvé - aucune {list_type} appliquée")
        return []

def load_exclusion_list(filepath):
    """Charge la liste des fichiers non traités à exclure."""
    return load_text_list(filepath, "liste d'exclusion")

def should_exclude_file(filename, exclusion_list):
    """Vérifie si un fichier doit être exclu.
    Le fichier NON_TRAITE contient uniquement la partie Auteur_Nom
    (sans le préfixe SOUSCORPUS_ et sans l'extension .xml).
    Ex: fichier = PEINTURE_Auteur_Nom.xml → on compare 'Auteur_Nom' à la liste
    """
    name_without_ext = filename.stem  # PEINTURE_Auteur_Nom
    # Supprimer le préfixe du sous-corpus (première partie avant '_')
    parts = name_without_ext.split('_', 1)
    author_name = parts[1] if len(parts) > 1 else name_without_ext  # Auteur_Nom
    return author_name in exclusion_list

def load_corpus_data(directory, subcorpus, exclusion_list=None):
    """Charge les données XML d'un sous-corpus avec filtrage NON_TRAITE."""
    exclusion_list = exclusion_list or []
    all_xml_files = sorted(Path(directory).glob(f'{subcorpus}_*.xml'))

    print(f"\n{'─'*60}")
    print(f"Fichiers trouvés pour [{subcorpus}] : {len(all_xml_files)}")
    print(f"{'─'*60}")

    xml_files = []
    for f in all_xml_files:
        if should_exclude_file(f, exclusion_list):
            print(f"  ⊗ EXCLU    {f.name}")
        else:
            print(f"  ✓ INCLUS   {f.name}")
            xml_files.append(f)

    print(f"{'─'*60}")
    print(f"  → {len(xml_files)} chargé(s), {len(all_xml_files) - len(xml_files)} exclu(s)")
    print(f"{'─'*60}\n")

    all_tokens = []
    for xml_file in xml_files:
        tokens = extract_tokens_from_xml(xml_file)
        n = len(tokens)
        print(f"  📄 {xml_file.name:<50} {n:>8,} tokens")
        all_tokens.extend(tokens)

    df = pd.DataFrame(all_tokens)
    df_filtered = filter_dataframe(df)

    print(f"\n  → Tokens avant filtrage : {len(df):,}")
    print(f"  → Tokens après filtrage : {len(df_filtered):,}")

    return df_filtered


def load_named_entities(directory, subcorpus, exclusion_list=None):
    """Charge les entités nommées d'un sous-corpus avec filtrage NON_TRAITE."""
    exclusion_list = exclusion_list or []
    all_xml_files = sorted(Path(directory).glob(f'{subcorpus}_*.xml'))
    xml_files = [f for f in all_xml_files if not should_exclude_file(f, exclusion_list)]

    print(f"\n  Entités nommées [{subcorpus}] :")
    all_entities = []
    for xml_file in xml_files:
        entities = extract_named_entities_from_xml(xml_file)
        n = len(entities)
        print(f"  📄 {xml_file.name:<50} {n:>8,} entités")
        all_entities.extend(entities)

    return pd.DataFrame(all_entities)


# =============================================================================
# SÉLECTION DU SOUS-CORPUS
# Décommentez UNE seule ligne (ou plusieurs pour combiner)
# =============================================================================

SUBCORPUS = "PEINTURE"
# SUBCORPUS = "ARCHITECTURE"
# SUBCORPUS = "PERSPECTIVE"
# SUBCORPUS = ["PEINTURE", "ARCHITECTURE"]   # combinaison possible

# =============================================================================

print("\n" + "="*60)
print(f"SOUS-CORPUS SÉLECTIONNÉ : {SUBCORPUS}")
print("="*60 + "\n")

exclusion_list = load_exclusion_list(NON_TRAITE)

# Gestion corpus simple ou combiné
subcorpus_list = [SUBCORPUS] if isinstance(SUBCORPUS, str) else SUBCORPUS

df = pd.concat(
    [load_corpus_data(OUTPUT_DIR, sc, exclusion_list) for sc in subcorpus_list],
    ignore_index=True
)
entities = pd.concat(
    [load_named_entities(OUTPUT_DIR, sc, exclusion_list) for sc in subcorpus_list],
    ignore_index=True
)

print(f"\n✓ Données chargées :")
print(f"  - Tokens  : {len(df):,}")
print(f"  - Entités : {len(entities):,}")


SOUS-CORPUS SÉLECTIONNÉ : PEINTURE

✓ Liste d'exclusion chargée : 40 entrées

────────────────────────────────────────────────────────────
Fichiers trouvés pour [PEINTURE] : 16
────────────────────────────────────────────────────────────
  ✓ INCLUS   PEINTURE_Daret_VieRaphael.xml
  ✓ INCLUS   PEINTURE_DupuyDuGrez_TraitePeinture.xml
  ⊗ EXCLU    PEINTURE_Felibien_Entretiens1.xml
  ⊗ EXCLU    PEINTURE_Felibien_Entretiens2.xml
  ⊗ EXCLU    PEINTURE_Felibien_Entretiens3.xml
  ⊗ EXCLU    PEINTURE_Felibien_Entretiens4.xml
  ⊗ EXCLU    PEINTURE_Felibien_Entretiens5.xml
  ✓ INCLUS   PEINTURE_Lomazzo_TraicteProportion.xml
  ✓ INCLUS   PEINTURE_Monier_HistoireArtsRapportDessein.xml
  ✓ INCLUS   PEINTURE_Pader_LaPeintureParlante.xml
  ✓ INCLUS   PEINTURE_Pader_SongeEnigmatique.xml
  ✓ INCLUS   PEINTURE_Piles_AbregeViePeintres.xml
  ✓ INCLUS   PEINTURE_Piles_ConversationsConnaissancePeinture.xml
  ✓ INCLUS   PEINTURE_Piles_CoursPeinture.xml
  ✓ INCLUS   PEINTURE_Piles_DialogueColoris.xml
  ✓ INCL

## objectName – Extraction et analyses

In [6]:
"""
## Extraction des objectName
Dans les XML du corpus, <objectName> est une balise inline ENTRE les <w>,
non un conteneur de <w>. On reconstruit la position en trouvant l'index du
<w> précédent dans le flux linéaire du document.
"""

def extract_objectnames_from_xml(xml_path):
    parser = etree.XMLParser(remove_blank_text=False)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()

    # Liste ORDONNÉE de tous les nœuds <w> du document
    all_w    = root.xpath('.//tei:w | .//w', namespaces=NSMAP)
    w_texts  = [(w.text or '').strip() for w in all_w]
    w_lemmas = [w.get('lemma', '').lower() for w in all_w]
    w_pos    = [w.get('pos', '') for w in all_w]
    w_id     = {id(w): i for i, w in enumerate(all_w)}

    results = []
    obj_elements = root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP)

    for obj in obj_elements:
        surface = ' '.join(obj.itertext()).strip()
        if not surface:
            continue
        ref = obj.get('ref', '').strip().lstrip('#')

        # Stratégie : chercher le <w> précédent dans le document
        # On remonte dans l'arbre pour trouver tous les <w> AVANT cet objectName
        # en utilisant l'ordre du document (preceding-sibling ou preceding axis)
        preceding_ws = obj.xpath(
            'preceding::tei:w | preceding::w', namespaces=NSMAP
        )
        if preceding_ws:
            last_preceding = preceding_ws[-1]
            pos_in_doc = w_id.get(id(last_preceding), -1)
            # La position de l'objectName est juste APRÈS le dernier <w> précédent
            insert_pos = pos_in_doc + 1 if pos_in_doc >= 0 else 0
        else:
            insert_pos = 0

        results.append({
            'fichier'   : xml_path.name,
            'ref'       : ref,
            'surface'   : surface,
            'insert_pos': insert_pos,   # position entre deux <w>
            'w_texts'   : w_texts,
            'w_lemmas'  : w_lemmas,
            'w_pos'     : w_pos,
        })
    return results


# Charger tous les fichiers du sous-corpus
all_xml_files    = sorted(Path(OUTPUT_DIR).glob(f'{SUBCORPUS}_*.xml'))
non_traite_list  = load_exclusion_list(NON_TRAITE)
xml_files        = [f for f in all_xml_files
                    if not should_exclude_file(f, non_traite_list)]

raw_objects = []
for xml_file in xml_files:
    raw_objects.extend(extract_objectnames_from_xml(xml_file))

# DataFrame léger
df_obj = pd.DataFrame([
    {'fichier': r['fichier'], 'ref': r['ref'], 'surface': r['surface'],
     'has_ref': bool(r['ref'])}
    for r in raw_objects
])

df_obj_id   = df_obj[df_obj['has_ref']].copy()
df_obj_noid = df_obj[~df_obj['has_ref']].copy()

print(f'✓ {len(df_obj):,} occurrences d\'objectName extraites')
print(f'  ├─ {len(df_obj_id):,} IDENTIFIÉES  (ref renseigné)'
      f' → {df_obj_id["ref"].nunique()} ref uniques')
print(f'  └─ {len(df_obj_noid):,} NON IDENTIFIÉES (ref vide)')
print(f'\n  Fichiers sources : {df_obj["fichier"].nunique()}')
print()
display(df_obj_id[['fichier', 'ref', 'surface']].head(10))


✓ Liste d'exclusion chargée : 40 entrées
✓ 1,441 occurrences d'objectName extraites
  ├─ 444 IDENTIFIÉES  (ref renseigné) → 317 ref uniques
  └─ 997 NON IDENTIFIÉES (ref vide)

  Fichiers sources : 10



,fichier,ref,surface
0,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaOddi,grand Autel de S. François de Peruge
1,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaOddi,Tableau d ' Assomption de la Vierge
2,PEINTURE_Daret_VieRaphael.xml,RaphaelPredelleOddi,marchepied de cet Autel
3,PEINTURE_Daret_VieRaphael.xml,RaphaelAnnonciationOddi,Annontiation
4,PEINTURE_Daret_VieRaphael.xml,RaphaelAdorationMagesOddi,l ' Adoration des Rois
5,PEINTURE_Daret_VieRaphael.xml,RaphaelPresentationTempleOddi,S. Siméon au Temple
6,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaSanNicola,premier Tableau qu ' il peignit de son inventi...
7,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaGavari,Crucifix à S. Dominique
8,PEINTURE_Daret_VieRaphael.xml,RaphaelSpozalio,petit Tableau des épousailles de la Vierge et ...
9,PEINTURE_Daret_VieRaphael.xml,LibreriaPiccolomini,Bibliotecque du Dome


In [7]:
"""
## Visualisation – Distribution des objectName IDENTIFIÉS par texte
Seules les occurrences avec ref sont utilisées ici.
"""

if df_obj_id.empty:
    print('⚠ Aucun objectName identifié. Cellule ignorée.')
else:
    def short_name(fname):
        return fname.replace(f'{SUBCORPUS}_', '').replace('.xml', '')

    TOP_N = 30
    top_refs = df_obj_id['ref'].value_counts().head(TOP_N).index
    df_top   = df_obj_id[df_obj_id['ref'].isin(top_refs)].copy()
    df_top['texte'] = df_top['fichier'].apply(short_name)

    # ── Viz 1 : Heatmap ref × texte ────────────────────────────────────────
    pivot = df_top.pivot_table(
        index='ref', columns='texte', values='surface',
        aggfunc='count', fill_value=0
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(max(14, len(pivot.columns) * 0.9),
                                     max(8, len(pivot) * 0.4)))
    sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.4,
                annot=(len(pivot) <= 20), fmt='d', ax=ax,
                cbar_kws={'label': "Nb d'occurrences"})
    ax.set_title(f'Distribution des objectName identifiés (top {TOP_N} ref) par texte',
                 fontsize=13, pad=14)
    ax.set_xlabel('Texte source', fontsize=10)
    ax.set_ylabel('Identifiant d\'œuvre (ref)', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    out1 = Path(VISUALIZATIONS_DIR) / 'objectname_heatmap_identifies.png'
    plt.savefig(out1, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out1}')

    # ── Viz 2 : Bar chart top ref ───────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(13, 5))
    counts = df_obj_id['ref'].value_counts().head(25)
    counts[::-1].plot(kind='barh', ax=ax2,
                      color=sns.color_palette('viridis', len(counts))[::-1])
    ax2.set_title('Top 25 – objectName identifiés les plus fréquents', fontsize=12)
    ax2.set_xlabel("Nb total d'occurrences")
    ax2.set_ylabel('Identifiant (ref)')
    plt.tight_layout()
    out2 = Path(VISUALIZATIONS_DIR) / 'objectname_top25_identifies.png'
    plt.savefig(out2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out2}')

    # ── Viz 3 : Identifiés vs non-identifiés par texte ─────────────────────
    counts_id   = df_obj[df_obj['has_ref']].groupby('fichier').size().rename('identifiés')
    counts_noid = df_obj[~df_obj['has_ref']].groupby('fichier').size().rename('non identifiés')
    df_ratio = pd.concat([counts_id, counts_noid], axis=1).fillna(0).astype(int)
    df_ratio.index = df_ratio.index.map(short_name)
    df_ratio = df_ratio.sort_values('identifiés', ascending=False)

    fig3, ax3 = plt.subplots(figsize=(14, 5))
    df_ratio[['identifiés', 'non identifiés']].plot(
        kind='bar', ax=ax3, color=['steelblue', 'lightcoral'], edgecolor='white'
    )
    ax3.set_title('objectName identifiés vs non identifiés par texte', fontsize=12)
    ax3.set_xlabel('Texte source')
    ax3.set_ylabel("Nb d'occurrences")
    ax3.tick_params(axis='x', rotation=45, labelsize=8)
    ax3.legend()
    plt.tight_layout()
    out3 = Path(VISUALIZATIONS_DIR) / 'objectname_ratio_identifies.png'
    plt.savefig(out3, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out3}')


✓ Sauvegardé : humanstica_output\objectname_heatmap_identifies.png
✓ Sauvegardé : humanstica_output\objectname_top25_identifies.png
✓ Sauvegardé : humanstica_output\objectname_ratio_identifies.png


In [8]:
"""
## Visualisation – Distribution des objectName IDENTIFIÉS par texte
On ne travaille ici qu'avec les occurrences qui possèdent un ref (identifiant
unique d'œuvre). Les objectName sans ref sont exclus de ces visualisations.
"""

if df_obj_id.empty:
    print('⚠ Aucun objectName identifié (ref vide pour tous). Cellule ignorée.')
else:
    TOP_N = 30
    top_refs = df_obj_id['ref'].value_counts().head(TOP_N).index
    df_top   = df_obj_id[df_obj_id['ref'].isin(top_refs)]

    # Raccourcir les noms de fichiers
    def short_name(fname):
        return fname.replace(f'{SUBCORPUS}_', '').replace('.xml', '')
    df_top = df_top.copy()
    df_top['texte'] = df_top['fichier'].apply(short_name)

    # ── Viz 1 : Heatmap ref × texte ────────────────────────────────────────
    pivot = df_top.pivot_table(
        index='ref', columns='texte', values='surface',
        aggfunc='count', fill_value=0
    )
    # Trier les lignes par fréquence totale décroissante
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(max(14, len(pivot.columns) * 0.9),
                                     max(8, len(pivot) * 0.4)))
    sns.heatmap(
        pivot, cmap='YlOrRd', linewidths=0.4,
        annot=(len(pivot) <= 20), fmt='d', ax=ax,
        cbar_kws={'label': "Nb d'occurrences"}
    )
    ax.set_title(
        f'Distribution des objectName identifiés (top {TOP_N} ref) par texte source',
        fontsize=13, pad=14
    )
    ax.set_xlabel('Texte source', fontsize=10)
    ax.set_ylabel('Identifiant d\'œuvre (ref)', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    out1 = Path(VISUALIZATIONS_DIR) / 'objectname_heatmap_identifies.png'
    plt.savefig(out1, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out1}')

    # ── Viz 2 : Bar chart top ref toutes sources confondues ────────────────
    fig2, ax2 = plt.subplots(figsize=(13, 5))
    counts = df_obj_id['ref'].value_counts().head(25)
    colors = sns.color_palette('viridis', len(counts))
    counts[::-1].plot(kind='barh', ax=ax2, color=colors[::-1])
    ax2.set_title('Top 25 – objectName identifiés les plus fréquents (toutes sources)',
                  fontsize=12)
    ax2.set_xlabel("Nb total d'occurrences")
    ax2.set_ylabel('Identifiant (ref)')
    plt.tight_layout()
    out2 = Path(VISUALIZATIONS_DIR) / 'objectname_top25_identifies.png'
    plt.savefig(out2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out2}')

    # ── Viz 3 : Proportion identifiés vs non-identifiés par texte ──────────
    counts_id   = df_obj[df_obj['has_ref']].groupby('fichier').size().rename('identifiés')
    counts_noid = df_obj[~df_obj['has_ref']].groupby('fichier').size().rename('non identifiés')
    df_ratio = pd.concat([counts_id, counts_noid], axis=1).fillna(0).astype(int)
    df_ratio['texte'] = df_ratio.index.map(short_name)
    df_ratio = df_ratio.sort_values('identifiés', ascending=False).set_index('texte')

    fig3, ax3 = plt.subplots(figsize=(14, 5))
    df_ratio[['identifiés', 'non identifiés']].plot(
        kind='bar', ax=ax3, color=['steelblue', 'lightcoral'], edgecolor='white'
    )
    ax3.set_title('objectName identifiés vs non identifiés par texte', fontsize=12)
    ax3.set_xlabel('Texte source')
    ax3.set_ylabel('Nb d\'occurrences')
    ax3.tick_params(axis='x', rotation=45, labelsize=8)
    ax3.legend()
    plt.tight_layout()
    out3 = Path(VISUALIZATIONS_DIR) / 'objectname_ratio_identifies.png'
    plt.savefig(out3, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Sauvegardé : {out3}')


✓ Sauvegardé : humanstica_output\objectname_heatmap_identifies.png
✓ Sauvegardé : humanstica_output\objectname_top25_identifies.png
✓ Sauvegardé : humanstica_output\objectname_ratio_identifies.png


In [16]:
"""
## Analyse KWIC – Contexte lexical des objectName
<objectName> est une balise inline entre les <w> : on utilise insert_pos
(index du premier <w> suivant l'objectName) pour centrer la fenêtre.
"""

WINDOW_SIZE = 3

def build_kwic(raw_objects, window=WINDOW_SIZE):
    rows = []
    for r in raw_objects:
        wt  = r['w_texts']
        wl  = r['w_lemmas']
        wp  = r['w_pos']
        pos = r['insert_pos']   # index juste après l'objectName dans la liste des <w>

        # Contexte gauche  = tokens AVANT l'objectName
        left_start  = max(0, pos - window)
        left_ctx    = ' '.join(wt[left_start:pos])

        # Contexte droit = tokens APRÈS l'objectName
        right_end   = min(len(wt), pos + window)
        right_ctx   = ' '.join(wt[pos:right_end])

        rows.append({
            'fichier'    : r['fichier'],
            'ref'        : r['ref'],
            'has_ref'    : bool(r['ref']),
            'gauche'     : left_ctx,
            'mot_cle'    : r['surface'],
            'droite'     : right_ctx,
            'left_start' : left_start,
            'right_end'  : right_end,
            'w_lemmas'   : wl,
            'w_pos'      : wp,
        })
    cols = ['fichier','ref','has_ref','gauche','mot_cle','droite',
            'left_start','right_end','w_lemmas','w_pos']
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)


df_kwic = build_kwic(raw_objects, window=WINDOW_SIZE)

if df_kwic.empty:
    print('⚠ KWIC vide : aucun objectName extrait.')
else:
    print(f'✓ {len(df_kwic):,} lignes KWIC | fenêtre ±{WINDOW_SIZE} tokens')
    print(f'  ├─ avec ref  : {df_kwic["has_ref"].sum():,}')
    print(f'  └─ sans ref  : {(~df_kwic["has_ref"]).sum():,}')

    pd.set_option('display.max_colwidth', 55)
    display(df_kwic[['fichier', 'ref', 'gauche', 'mot_cle', 'droite']].head(800))

    # ── Nuage de mots du contexte ───────────────────────────────────────────
    STOPWORDS_COMBINED = STOPWORDS_FR | STOPWORDS_IT
    ctx_tokens = []
    for _, row in df_kwic.iterrows():
        for tok in (row['gauche'] + ' ' + row['droite']).lower().split():
            if tok not in STOPWORDS_COMBINED and len(tok) > 2 and tok.isalpha():
                ctx_tokens.append(tok)

    if ctx_tokens:
        wc = WordCloud(
            width=1400, height=600, background_color='white',
            max_words=150, colormap='plasma'
        ).generate_from_frequencies(Counter(ctx_tokens))
        fig, ax = plt.subplots(figsize=(16, 6))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Contexte lexical des objectName – nuage de mots (hors stopwords)',
                     fontsize=13)
        plt.tight_layout()
        out_wc = Path(VISUALIZATIONS_DIR) / 'objectname_kwic_wordcloud.png'
        plt.savefig(out_wc, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_wc}')


✓ 1,441 lignes KWIC | fenêtre ±3 tokens
  ├─ avec ref  : 444
  └─ sans ref  : 997


,fichier,ref,gauche,mot_cle,droite
0,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaOddi,ce fut au,grand Autel de S. François de Peruge,", où il"
1,PEINTURE_Daret_VieRaphael.xml,RaphaelPalaOddi,huile à un,Tableau d ' Assomption de la Vierge,", du dessein"
2,PEINTURE_Daret_VieRaphael.xml,RaphaelPredelleOddi,et dans le,marchepied de cet Autel,il peignit rois
3,PEINTURE_Daret_VieRaphael.xml,RaphaelAnnonciationOddi,"sont , une",Annontiation,", et ,"
4,PEINTURE_Daret_VieRaphael.xml,RaphaelAdorationMagesOddi,", une ,",l ' Adoration des Rois,"et , qui"
...,...,...,...,...,...
795,PEINTURE_Monier_HistoireArtsRapportDessein.xml,,porté à des,Tableaux qui sont dans la galerie des grands Jesuistes,qui ont du
796,PEINTURE_Monier_HistoireArtsRapportDessein.xml,,l ' admirable,"Tableau de Rafaël , qui represente sainte Cecile av...",. lors qu
797,PEINTURE_Monier_HistoireArtsRapportDessein.xml,,que de recevoir,celui de la sainte Cecile,", et avoit"
798,PEINTURE_Monier_HistoireArtsRapportDessein.xml,,qui embelirent de,leurs Peintures,les Eglises de


✓ Sauvegardé : humanstica_output\objectname_kwic_wordcloud.png


In [17]:
"""
## Verbes dans le contexte KWIC des objectName
Extrait les lemmes verbaux (VERB / AUX / VER:*) dans la fenêtre contextuelle.
"""

if df_kwic.empty:
    print('⚠ KWIC vide – cellule ignorée.')
else:
    VERB_PREFIXES = ('VERB', 'AUX', 'VER')

    def extract_context_verbs(row):
        lemmas = row['w_lemmas']
        pos    = row['w_pos']
        verbs  = []
        for i in range(row['left_start'], min(row['right_end'], len(pos))):
            if any(pos[i].upper().startswith(p) for p in VERB_PREFIXES):
                lem = lemmas[i]
                if lem and len(lem) > 2 and lem.isalpha():
                    verbs.append(lem)
        return verbs

    df_kwic['verbes'] = df_kwic.apply(extract_context_verbs, axis=1)

    verb_rows = [
        {'ref': row['ref'], 'has_ref': row['has_ref'],
         'fichier': row['fichier'], 'verbe': v}
        for _, row in df_kwic.iterrows()
        for v in row['verbes']
    ]
    df_verbs = pd.DataFrame(verb_rows) if verb_rows else pd.DataFrame(
        columns=['ref', 'has_ref', 'fichier', 'verbe']
    )

    print(f'✓ {len(df_verbs):,} occurrences verbales dans les contextes KWIC')
    print(f'  {df_verbs["verbe"].nunique()} lemmes verbaux distincts')

    if df_verbs.empty:
        print('⚠ Aucun verbe trouvé – vérifiez les valeurs POS dans vos XML.')
    else:
        # ── Viz 1 : Top 30 verbes globaux ──────────────────────────────────
        top_v = df_verbs['verbe'].value_counts().head(30)
        fig, ax = plt.subplots(figsize=(13, 6))
        top_v[::-1].plot(kind='barh', ax=ax,
                         color=sns.color_palette('mako', len(top_v))[::-1])
        ax.set_title('Verbes dans le contexte des objectName', fontsize=12)
        ax.set_xlabel('Fréquence')
        plt.tight_layout()
        out_v1 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_top30.png'
        plt.savefig(out_v1, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_v1}')

        # ── Viz 2 : Heatmap verbe × ref (identifiés seulement) ─────────────
        df_verbs_id = df_verbs[df_verbs['has_ref']]
        if not df_verbs_id.empty:
            TOP_R, TOP_V = 20, 20
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8, len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 3 : Profil verbal des 5 objectName identifiés les + fréquents
        if not df_verbs_id.empty:
            top5 = df_verbs_id['ref'].value_counts().head(5).index.tolist()
            n    = len(top5)
            fig3, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
            if n == 1:
                axes = [axes]
            for ax_i, ref_id in zip(axes, top5):
                sub_ref = (df_verbs_id[df_verbs_id['ref'] == ref_id]['verbe']
                           .value_counts().head(10))
                sub_ref[::-1].plot(
                    kind='barh', ax=ax_i,
                    color=sns.color_palette('rocket', len(sub_ref))[::-1]
                )
                ax_i.set_title(ref_id[:28], fontsize=9)
                ax_i.set_xlabel('Fréq.')
                ax_i.tick_params(labelsize=8)
            fig3.suptitle(
                'Profil verbal des 5 objectName identifiés les plus fréquents',
                fontsize=11, y=1.02
            )
            plt.tight_layout()
            out_v3 = Path(VISUALIZATIONS_DIR) / 'objectname_profil_verbal.png'
            plt.savefig(out_v3, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'✓ Sauvegardé : {out_v3}')


✓ 1,139 occurrences verbales dans les contextes KWIC
  231 lemmes verbaux distincts
✓ Sauvegardé : humanstica_output\objectname_verbes_top30.png
✓ Sauvegardé : humanstica_output\objectname_verbes_heatmap.png
✓ Sauvegardé : humanstica_output\objectname_profil_verbal.png


### Verbes dans le contexte KWIC des objectName

In [44]:
"""
## Verbes dans le contexte KWIC des objectName
Extrait les lemmes verbaux (VERB / AUX / VER:*) dans la fenêtre contextuelle.
"""

if df_kwic.empty:
    print('⚠ KWIC vide – cellule ignorée.')
else:
    VERB_PREFIXES = ('VERB', 'AUX', 'VER')

    def extract_context_verbs(row):
        lemmas = row['w_lemmas']
        pos    = row['w_pos']
        verbs  = []
        for i in range(row['left_start'], min(row['right_end'], len(pos))):
            if any(pos[i].upper().startswith(p) for p in VERB_PREFIXES):
                lem = lemmas[i]
                if lem and len(lem) > 2 and lem.isalpha():
                    verbs.append(lem)
        return verbs

    df_kwic['verbes'] = df_kwic.apply(extract_context_verbs, axis=1)

    verb_rows = [
        {'ref': row['ref'], 'has_ref': row['has_ref'],
         'fichier': row['fichier'], 'verbe': v}
        for _, row in df_kwic.iterrows()
        for v in row['verbes']
    ]
    df_verbs = pd.DataFrame(verb_rows) if verb_rows else pd.DataFrame(
        columns=['ref', 'has_ref', 'fichier', 'verbe']
    )

    print(f'✓ {len(df_verbs):,} occurrences verbales dans les contextes KWIC')
    print(f'  {df_verbs["verbe"].nunique()} lemmes verbaux distincts')

    if df_verbs.empty:
        print('⚠ Aucun verbe trouvé – vérifiez les valeurs POS dans vos XML.')
    else:
        # ── Viz 1 : Top 30 verbes globaux ──────────────────────────────────
        top_v = df_verbs['verbe'].value_counts().head(100)
        fig, ax = plt.subplots(figsize=(13, 6))
        top_v[::-1].plot(kind='barh', ax=ax,
                         color=sns.color_palette('mako', len(top_v))[::-1])
        ax.set_title('Top 100 – Verbes dans le contexte des objectName', fontsize=12)
        ax.set_xlabel('Fréquence')
        plt.tight_layout()
        out_v1 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_top100.png'
        plt.savefig(out_v1, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_v1}')

        # ── Viz 2 : Heatmap verbe × ref (identifiés seulement) ─────────────
        df_verbs_id = df_verbs[df_verbs['has_ref']]
        if not df_verbs_id.empty:
            TOP_R, TOP_V = 100, 100
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8, len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 2b : Heatmap verbe × objectName (tous, identifiés ou non) ─────────────
    
            TOP_R, TOP_V = 100, 100
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8, len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap_general.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 3 : Profil verbal des 5 objectName identifiés les + fréquents
        if not df_verbs_id.empty:
            top5 = df_verbs_id['ref'].value_counts().head(5).index.tolist()
            n    = len(top5)
            fig3, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
            if n == 1:
                axes = [axes]
            for ax_i, ref_id in zip(axes, top5):
                sub_ref = (df_verbs_id[df_verbs_id['ref'] == ref_id]['verbe']
                           .value_counts().head(10))
                sub_ref[::-1].plot(
                    kind='barh', ax=ax_i,
                    color=sns.color_palette('rocket', len(sub_ref))[::-1]
                )
                ax_i.set_title(ref_id[:28], fontsize=9)
                ax_i.set_xlabel('Fréq.')
                ax_i.tick_params(labelsize=8)
            fig3.suptitle(
                'Profil verbal des 5 objectName identifiés les plus fréquents',
                fontsize=11, y=1.02
            )
            plt.tight_layout()
            out_v3 = Path(VISUALIZATIONS_DIR) / 'objectname_profil_verbal.png'
            plt.savefig(out_v3, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'✓ Sauvegardé : {out_v3}')


✓ 3,639 occurrences verbales dans les contextes KWIC
  838 lemmes verbaux distincts
✓ Sauvegardé : humanstica_output\objectname_verbes_top100.png
✓ Sauvegardé : humanstica_output\objectname_verbes_heatmap.png
✓ Sauvegardé : humanstica_output\objectname_verbes_heatmap_general.png
✓ Sauvegardé : humanstica_output\objectname_profil_verbal.png


# TEST AVEC FREEM

In [29]:

"""
Configuration initiale et imports
Exécuter cette cellule en premier
"""
%matplotlib inline
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import matplotlib
matplotlib.use('Agg')  # Changer en 'inline' pour Jupyter

import spacy
from lxml import etree
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Configuration des chemins
OUTPUT_DIR = "output_lemmes_freem"
VISUALIZATIONS_DIR = "humanstica_output_freem"
TEI_NS = "http://www.tei-c.org/ns/1.0"
NSMAP = {'tei': TEI_NS}

# Créer les dossiers de sortie
Path(VISUALIZATIONS_DIR).mkdir(exist_ok=True)

print("✓ Configuration chargée")


✓ Configuration chargée


In [28]:
"""
Charger les modèles spaCy et les stopwords
"""

try:
    nlp_fr = spacy.load('fr_core_news_sm')
    nlp_it = spacy.load('it_core_news_sm')
    
    more_stopwords_fra = ['ay','cy','dict','-','pl','a.','b.','c.','d.','e.','f.','g.','h.','i.','j.','k.','l.','m.','n.','o.','p.','q.','r.','s.','t.','u.','v.','w.','x.','y.','z.','-là','apre','chose', 'faict','ester','aller','autant','celer','trop','beaucoup','estr','étoit','temp','rien','avoit', 'aucun','tre', 'ilz', 'aprer', 'estoier', 'faire', 'estre', 'fut', 'bien', 'estoit', 'mesme', 'luy', 'faut', 'faisoit', 'fit', 'costé', 'non', '\'', 'l', 'd', 'qu', 'j', 'n', 'c', 's']
    more_stopwords_ita = ['gl','quell','a.','b.','c.','d.','e.','f.','g.','h.','i.','j.','k.','l.','m.','n.','o.','p.','q.','r.','s.','t.','u.','v.','w.','x.','y.','z.','ch','cose', 'altra', 'como','esso','nostr','stare','lo','si','lor','andare','qual', 'altre', 'ciò', 'da', 'il', 'da il', 'di il', 'il di', 'su il', 'con il', 'lo', 'et', 'l', '\'', 's.', 'in il', 'o', 'de', 'd', 'a il']
    
    # Ajouter les stopwords personnalisés
    for item in more_stopwords_fra:
        nlp_fr.Defaults.stop_words.add(item)
    
    for item in more_stopwords_ita:
        nlp_it.Defaults.stop_words.add(item)
    
    # Récupérer les ensembles de stopwords
    STOPWORDS_FR = nlp_fr.Defaults.stop_words
    STOPWORDS_IT = nlp_it.Defaults.stop_words
    
    print("✓ Stopwords chargés depuis spaCy")
    print(f"  - Français: {len(STOPWORDS_FR)} stopwords")
    print(f"  - Italien: {len(STOPWORDS_IT)} stopwords")
    
except Exception as e:
    print(f"⚠ Impossible de charger les modèles spaCy: {e}")
    print("  Utilisation des stopwords par défaut")
    STOPWORDS_FR = {'le', 'la', 'les', 'un', 'une', 'des', 'de', 'du', 'et', 'ou'}
    STOPWORDS_IT = {'il', 'lo', 'la', 'i', 'gli', 'le', 'un', 'uno', 'una', 'di'}

✓ Stopwords chargés depuis spaCy
  - Français: 578 stopwords
  - Italien: 677 stopwords


In [30]:

"""
Fonctions de base pour l'extraction et le filtrage
"""

def should_exclude_by_pos(pos):
    """Exclut ponctuation et nombres – style FREEM/TreeTagger.
    PONfbl, PONfrt, PONpxx, PONpdr → ponctuation
    PROcar, DETcar, ADJcar          → cardinaux/nombres
    """
    if not pos:
        return False
    excluded_prefixes = ('PON',)
    excluded_tags     = {'PROcar', 'DETcar', 'ADJcar'}
    return pos in excluded_tags or any(pos.startswith(p) for p in excluded_prefixes)

def filter_dataframe(df):
    """Filtre le dataframe pour enlever ponctuation et nombres selon POS."""
    return df[~df['pos'].apply(should_exclude_by_pos)]

def extract_tokens_from_xml(xml_path):
    """Extrait tous les tokens <w> d'un fichier XML annoté FREEM."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()

    tokens_data = []
    w_elements  = root.xpath('.//tei:w | .//w', namespaces=NSMAP)

    for w in w_elements:
        token_text = w.text if w.text else ''
        lemma      = w.get('lemma', '')
        pos        = w.get('pos', '')

        if token_text.strip() and lemma:
            tokens_data.append({
                'token': token_text.strip().lower(),
                'lemma': lemma.lower(),
                'pos'  : pos          # ex: "VERcjg", "NOMcom", "ADJqua"…
            })

    return tokens_data

def extract_named_entities_from_xml(xml_path):
    """Extrait les entités nommées (persName et placeName) d'un fichier XML."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()
    
    entities = []
    
    # Extraire les persName
    pers_elements = root.xpath('.//tei:persName | .//persName', namespaces=NSMAP)
    for pers in pers_elements:
        text = ''.join(pers.itertext()).strip()
        ref = pers.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'personne'
            })
    
    # Extraire les placeName
    place_elements = root.xpath('.//tei:placeName | .//placeName', namespaces=NSMAP)
    for place in place_elements:
        text = ''.join(place.itertext()).strip()
        ref = place.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'lieu'
            })
    
    # Extraire les placeName
    object_elements = root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP)
    for object in object_elements:
        text = ''.join(object.itertext()).strip()
        ref = object.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref,
                'type': 'object'
            })
    
    return entities

def calculate_statistics(df, language):
    """Calcule les statistiques descriptives du corpus."""
    stats = {
        'langue': language,
        'nb_tokens': len(df),
        'nb_tokens_uniques': df['token'].nunique(),
        'nb_lemmes_uniques': df['lemma'].nunique(),
        'ratio_diversite': df['lemma'].nunique() / len(df) if len(df) > 0 else 0,
        'longueur_moyenne_token': df['token'].str.len().mean()
    }
    return stats

def extract_ngrams(df, n=2):
    """Extrait les n-grammes d'un corpus."""
    lemmas = df['lemma'].values
    ngrams = []
    
    for i in range(len(lemmas) - n + 1):
        ngram = tuple(lemmas[i:i+n])
        ngrams.append(ngram)
    
    return Counter(ngrams)

print("✓ Fonctions utilitaires chargées")



✓ Fonctions utilitaires chargées


In [31]:
"""
CHARGEMENT DES DONNÉES
Charge tous les fichiers XML du dossier output_lemmes_freem.
"""

def should_exclude_by_pos(pos):
    """Exclut ponctuation et nombres – style FREEM/TreeTagger."""
    if not pos:
        return False
    excluded_tags     = {'PROcar', 'DETcar', 'ADJcar'}
    excluded_prefixes = ('PON',)
    return pos in excluded_tags or any(pos.startswith(p) for p in excluded_prefixes)

def filter_dataframe(df):
    """Filtre le dataframe pour enlever ponctuation et nombres."""
    return df[~df['pos'].apply(should_exclude_by_pos)]

def extract_tokens_from_xml(xml_path):
    """Extrait tous les tokens <w> d'un fichier XML annoté FREEM."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()

    tokens_data = []
    w_elements  = root.xpath('.//tei:w | .//w', namespaces=NSMAP)

    for w in w_elements:
        token_text = w.text if w.text else ''
        lemma      = w.get('lemma', '')
        pos        = w.get('pos', '')

        if token_text.strip() and lemma:
            tokens_data.append({
                'token'  : token_text.strip().lower(),
                'lemma'  : lemma.lower(),
                'pos'    : pos,
                'fichier': xml_path.name,
            })

    return tokens_data

def extract_named_entities_from_xml(xml_path):
    """Extrait persName, placeName et objectName d'un fichier XML."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()

    entities = []

    for pers in root.xpath('.//tei:persName | .//persName', namespaces=NSMAP):
        text = ''.join(pers.itertext()).strip()
        if text:
            entities.append({'text': text, 'ref': pers.get('ref', ''), 'type': 'personne'})

    for place in root.xpath('.//tei:placeName | .//placeName', namespaces=NSMAP):
        text = ''.join(place.itertext()).strip()
        if text:
            entities.append({'text': text, 'ref': place.get('ref', ''), 'type': 'lieu'})

    for obj in root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP):
        text = ''.join(obj.itertext()).strip()
        if text:
            entities.append({'text': text, 'ref': obj.get('ref', ''), 'type': 'object'})

    return entities

def load_corpus_data(directory):
    """Charge tous les fichiers XML du dossier."""
    xml_files = sorted(Path(directory).glob('*.xml'))

    print(f"\n{'─'*60}")
    print(f"Fichiers trouvés : {len(xml_files)}")
    print(f"{'─'*60}")

    all_tokens = []
    for xml_file in xml_files:
        tokens = extract_tokens_from_xml(xml_file)
        print(f"  📄 {xml_file.name:<50} {len(tokens):>8,} tokens")
        all_tokens.extend(tokens)

    df          = pd.DataFrame(all_tokens)
    df_filtered = filter_dataframe(df)

    print(f"{'─'*60}")
    print(f"  → Tokens avant filtrage : {len(df):,}")
    print(f"  → Tokens après filtrage : {len(df_filtered):,}")

    return df_filtered

def load_named_entities(directory):
    """Charge les entités nommées de tous les fichiers XML."""
    xml_files   = sorted(Path(directory).glob('*.xml'))
    all_entities = []
    print(f"\n  Entités nommées :")
    for xml_file in xml_files:
        entities = extract_named_entities_from_xml(xml_file)
        print(f"  📄 {xml_file.name:<50} {len(entities):>8,} entités")
        all_entities.extend(entities)
    return pd.DataFrame(all_entities)

# ── Chargement ──────────────────────────────────────────────────────────────
print("=" * 60)
print("CHARGEMENT DU CORPUS")
print("=" * 60)

df       = load_corpus_data(OUTPUT_DIR)
entities = load_named_entities(OUTPUT_DIR)

print(f"\n✓ Données chargées :")
print(f"  - Tokens  : {len(df):,}")
print(f"  - Entités : {len(entities):,}")

CHARGEMENT DU CORPUS

────────────────────────────────────────────────────────────
Fichiers trouvés : 11
────────────────────────────────────────────────────────────
  📄 Daret_VieRaphael.xml                                 10,306 tokens
  📄 DupuyDuGrez_TraitePeinture.xml                      116,734 tokens
  📄 Lomazzo_TraicteProportion.xml                        66,985 tokens
  📄 Monier_HistoireArtsRapportDessein.xml                64,642 tokens
  📄 Pader_LaPeintureParlante.xml                         24,542 tokens
  📄 Pader_SongeEnigmatique.xml                           15,780 tokens
  📄 Piles_AbregeViePeintres.xml                         119,413 tokens
  📄 Piles_ConversationsConnaissancePeinture.xml          52,526 tokens
  📄 Piles_CoursPeinture.xml                              86,691 tokens
  📄 Piles_DialogueColoris.xml                            12,959 tokens
  📄 Vinci_TraitePeinture_fra.xml                         65,279 tokens
─────────────────────────────────────────────────────

In [32]:
"""
## Extraction des objectName
<objectName> est une balise inline ENTRE les <w>.
On reconstruit la position en trouvant l'index du <w> précédent.
"""

def extract_objectnames_from_xml(xml_path):
    parser = etree.XMLParser(remove_blank_text=False)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()

    all_w    = root.xpath('.//tei:w | .//w', namespaces=NSMAP)
    w_texts  = [(w.text or '').strip() for w in all_w]
    w_lemmas = [w.get('lemma', '').lower() for w in all_w]
    w_pos    = [w.get('pos', '') for w in all_w]
    w_id     = {id(w): i for i, w in enumerate(all_w)}

    results      = []
    obj_elements = root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP)

    for obj in obj_elements:
        surface = ' '.join(obj.itertext()).strip()
        if not surface:
            continue
        ref = obj.get('ref', '').strip().lstrip('#')

        preceding_ws = obj.xpath('preceding::tei:w | preceding::w', namespaces=NSMAP)
        if preceding_ws:
            last_preceding = preceding_ws[-1]
            pos_in_doc     = w_id.get(id(last_preceding), -1)
            insert_pos     = pos_in_doc + 1 if pos_in_doc >= 0 else 0
        else:
            insert_pos = 0

        results.append({
            'fichier'   : xml_path.name,
            'ref'       : ref,
            'surface'   : surface,
            'insert_pos': insert_pos,
            'w_texts'   : w_texts,
            'w_lemmas'  : w_lemmas,
            'w_pos'     : w_pos,
        })
    return results

# ── Chargement ──────────────────────────────────────────────────────────────
xml_files   = sorted(Path(OUTPUT_DIR).glob('*.xml'))
raw_objects = []
for xml_file in xml_files:
    raw_objects.extend(extract_objectnames_from_xml(xml_file))

df_obj = pd.DataFrame([
    {'fichier': r['fichier'], 'ref': r['ref'], 'surface': r['surface'],
     'has_ref': bool(r['ref'])}
    for r in raw_objects
])

df_obj_id   = df_obj[df_obj['has_ref']].copy()
df_obj_noid = df_obj[~df_obj['has_ref']].copy()

print(f'✓ {len(df_obj):,} occurrences d\'objectName extraites')
print(f'  ├─ {len(df_obj_id):,} IDENTIFIÉES  ({df_obj_id["ref"].nunique()} ref uniques)')
print(f'  └─ {len(df_obj_noid):,} NON IDENTIFIÉES')
print(f'\n  Fichiers sources : {df_obj["fichier"].nunique()}')
display(df_obj_id[['fichier', 'ref', 'surface']].head(10))

✓ 1,441 occurrences d'objectName extraites
  ├─ 444 IDENTIFIÉES  (317 ref uniques)
  └─ 997 NON IDENTIFIÉES

  Fichiers sources : 10


,fichier,ref,surface
0,Daret_VieRaphael.xml,RaphaelPalaOddi,grand Autel de S . François de Peruge
1,Daret_VieRaphael.xml,RaphaelPalaOddi,Tableau d ' Assomption de la Vierge
2,Daret_VieRaphael.xml,RaphaelPredelleOddi,marchepied de cet Autel
3,Daret_VieRaphael.xml,RaphaelAnnonciationOddi,Annontiation
4,Daret_VieRaphael.xml,RaphaelAdorationMagesOddi,l ' Adoration des Rois
5,Daret_VieRaphael.xml,RaphaelPresentationTempleOddi,S . Siméon au Temple
6,Daret_VieRaphael.xml,RaphaelPalaSanNicola,premier Tableau qu ' il peignit de son invention fu...
7,Daret_VieRaphael.xml,RaphaelPalaGavari,Crucifix à S . Dominique
8,Daret_VieRaphael.xml,RaphaelSpozalio,petit Tableau des épousailles de la Vierge et Saint...
9,Daret_VieRaphael.xml,LibreriaPiccolomini,Bibliotecque du Dome


In [33]:
"""
## Analyse KWIC – Contexte lexical des objectName
"""

WINDOW_SIZE = 3

def build_kwic(raw_objects, window=WINDOW_SIZE):
    rows = []
    for r in raw_objects:
        wt  = r['w_texts']
        wl  = r['w_lemmas']
        wp  = r['w_pos']
        pos = r['insert_pos']

        left_start = max(0, pos - window)
        left_ctx   = ' '.join(wt[left_start:pos])
        right_end  = min(len(wt), pos + window)
        right_ctx  = ' '.join(wt[pos:right_end])

        rows.append({
            'fichier'   : r['fichier'],
            'ref'       : r['ref'],
            'has_ref'   : bool(r['ref']),
            'gauche'    : left_ctx,
            'mot_cle'   : r['surface'],
            'droite'    : right_ctx,
            'left_start': left_start,
            'right_end' : right_end,
            'w_lemmas'  : wl,
            'w_pos'     : wp,
        })
    cols = ['fichier','ref','has_ref','gauche','mot_cle','droite',
            'left_start','right_end','w_lemmas','w_pos']
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)

df_kwic = build_kwic(raw_objects, window=WINDOW_SIZE)

if df_kwic.empty:
    print('⚠ KWIC vide : aucun objectName extrait.')
else:
    print(f'✓ {len(df_kwic):,} lignes KWIC | fenêtre ±{WINDOW_SIZE} tokens')
    print(f'  ├─ avec ref  : {df_kwic["has_ref"].sum():,}')
    print(f'  └─ sans ref  : {(~df_kwic["has_ref"]).sum():,}')

    pd.set_option('display.max_colwidth', 55)
    display(df_kwic[['fichier', 'ref', 'gauche', 'mot_cle', 'droite']].head(800))

    # ── Nuage de mots du contexte ───────────────────────────────────────────
    ctx_tokens = []
    for _, row in df_kwic.iterrows():
        for tok in (row['gauche'] + ' ' + row['droite']).lower().split():
            if tok not in STOPWORDS_FR and len(tok) > 2 and tok.isalpha():
                ctx_tokens.append(tok)

    if ctx_tokens:
        wc = WordCloud(
            width=1400, height=600, background_color='white',
            max_words=150, colormap='plasma'
        ).generate_from_frequencies(Counter(ctx_tokens))
        fig, ax = plt.subplots(figsize=(16, 6))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Contexte lexical des objectName – nuage de mots (hors stopwords)',
                     fontsize=13)
        plt.tight_layout()
        out_wc = Path(VISUALIZATIONS_DIR) / 'objectname_kwic_wordcloud.png'
        plt.savefig(out_wc, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_wc}')

✓ 1,441 lignes KWIC | fenêtre ±3 tokens
  ├─ avec ref  : 444
  └─ sans ref  : 997


,fichier,ref,gauche,mot_cle,droite
0,Daret_VieRaphael.xml,RaphaelPalaOddi,ce fut au,grand Autel de S . François de Peruge,", où il"
1,Daret_VieRaphael.xml,RaphaelPalaOddi,huile à un,Tableau d ' Assomption de la Vierge,", du dessein"
2,Daret_VieRaphael.xml,RaphaelPredelleOddi,et dans le,marchepied de cet Autel,il peignit rois
3,Daret_VieRaphael.xml,RaphaelAnnonciationOddi,"sont , une",Annontiation,", et ,"
4,Daret_VieRaphael.xml,RaphaelAdorationMagesOddi,", une ,",l ' Adoration des Rois,"et , qui"
...,...,...,...,...,...
795,Monier_HistoireArtsRapportDessein.xml,,porté à des,Tableaux qui sont dans la galerie des grands Jesuistes,qui ont du
796,Monier_HistoireArtsRapportDessein.xml,,l ' admirable,"Tableau de Rafaël , qui represente sainte Cecile av...",. lors qu
797,Monier_HistoireArtsRapportDessein.xml,,que de recevoir,celui de la sainte Cecile,", et avoit"
798,Monier_HistoireArtsRapportDessein.xml,,qui embelirent de,leurs Peintures,les Eglises de


✓ Sauvegardé : humanstica_output_freem\objectname_kwic_wordcloud.png


In [34]:
"""
## Verbes dans le contexte KWIC des objectName
Style d'annotation FREEM : VERcjg, VERinf, VERppe, VERppa…
"""

if df_kwic.empty:
    print('⚠ KWIC vide – cellule ignorée.')
else:
    VERB_PREFIXES = ('VER', 'AUX')

    def extract_context_verbs(row):
        lemmas = row['w_lemmas']
        pos    = row['w_pos']
        verbs  = []
        for i in range(row['left_start'], min(row['right_end'], len(pos))):
            if any(pos[i].upper().startswith(p) for p in VERB_PREFIXES):
                lem = lemmas[i]
                if lem and len(lem) > 2 and lem.isalpha():
                    verbs.append(lem)
        return verbs

    df_kwic['verbes'] = df_kwic.apply(extract_context_verbs, axis=1)

    verb_rows = [
        {'ref': row['ref'], 'has_ref': row['has_ref'],
         'fichier': row['fichier'], 'verbe': v}
        for _, row in df_kwic.iterrows()
        for v in row['verbes']
    ]
    df_verbs = pd.DataFrame(verb_rows) if verb_rows else pd.DataFrame(
        columns=['ref', 'has_ref', 'fichier', 'verbe']
    )

    print(f'✓ {len(df_verbs):,} occurrences verbales dans les contextes KWIC')
    print(f'  {df_verbs["verbe"].nunique()} lemmes verbaux distincts')

    if df_verbs.empty:
        print('⚠ Aucun verbe trouvé – vérifiez les valeurs POS dans vos XML.')
    else:
        # ── Viz 1 : Top 30 verbes globaux ──────────────────────────────────
        top_v = df_verbs['verbe'].value_counts().head(30)
        fig, ax = plt.subplots(figsize=(13, 6))
        top_v[::-1].plot(kind='barh', ax=ax,
                         color=sns.color_palette('mako', len(top_v))[::-1])
        ax.set_title('Verbes dans le contexte des objectName', fontsize=12)
        ax.set_xlabel('Fréquence')
        plt.tight_layout()
        out_v1 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_top30.png'
        plt.savefig(out_v1, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_v1}')

        # ── Viz 2 : Heatmap verbe × ref (identifiés seulement) ─────────────
        df_verbs_id = df_verbs[df_verbs['has_ref']]
        if not df_verbs_id.empty:
            TOP_R, TOP_V = 20, 20
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8,  len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 3 : Profil verbal des 5 objectName les + fréquents ─────────
        if not df_verbs_id.empty:
            top5  = df_verbs_id['ref'].value_counts().head(5).index.tolist()
            n     = len(top5)
            fig3, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
            if n == 1:
                axes = [axes]
            for ax_i, ref_id in zip(axes, top5):
                sub_ref = (df_verbs_id[df_verbs_id['ref'] == ref_id]['verbe']
                           .value_counts().head(10))
                sub_ref[::-1].plot(
                    kind='barh', ax=ax_i,
                    color=sns.color_palette('rocket', len(sub_ref))[::-1]
                )
                ax_i.set_title(ref_id[:28], fontsize=9)
                ax_i.set_xlabel('Fréq.')
                ax_i.tick_params(labelsize=8)
            fig3.suptitle(
                'Profil verbal des 5 objectName identifiés les plus fréquents',
                fontsize=11, y=1.02
            )
            plt.tight_layout()
            out_v3 = Path(VISUALIZATIONS_DIR) / 'objectname_profil_verbal.png'
            plt.savefig(out_v3, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'✓ Sauvegardé : {out_v3}')

✓ 1,294 occurrences verbales dans les contextes KWIC
  194 lemmes verbaux distincts
✓ Sauvegardé : humanstica_output_freem\objectname_verbes_top30.png
✓ Sauvegardé : humanstica_output_freem\objectname_verbes_heatmap.png
✓ Sauvegardé : humanstica_output_freem\objectname_profil_verbal.png


In [ ]:
"""
## Verbes dans le contexte KWIC des objectName
Extrait les lemmes verbaux (VERB / AUX / VER:*) dans la fenêtre contextuelle.
"""

if df_kwic.empty:
    print('⚠ KWIC vide – cellule ignorée.')
else:
    VERB_PREFIXES = ('VERB', 'AUX', 'VER')

    def extract_context_verbs(row):
        lemmas = row['w_lemmas']
        pos    = row['w_pos']
        verbs  = []
        for i in range(row['left_start'], min(row['right_end'], len(pos))):
            if any(pos[i].upper().startswith(p) for p in VERB_PREFIXES):
                lem = lemmas[i]
                if lem and len(lem) > 2 and lem.isalpha():
                    verbs.append(lem)
        return verbs

    df_kwic['verbes'] = df_kwic.apply(extract_context_verbs, axis=1)

    verb_rows = [
        {'ref': row['ref'], 'has_ref': row['has_ref'],
         'fichier': row['fichier'], 'verbe': v}
        for _, row in df_kwic.iterrows()
        for v in row['verbes']
    ]
    df_verbs = pd.DataFrame(verb_rows) if verb_rows else pd.DataFrame(
        columns=['ref', 'has_ref', 'fichier', 'verbe']
    )

    print(f'✓ {len(df_verbs):,} occurrences verbales dans les contextes KWIC')
    print(f'  {df_verbs["verbe"].nunique()} lemmes verbaux distincts')

    if df_verbs.empty:
        print('⚠ Aucun verbe trouvé – vérifiez les valeurs POS dans vos XML.')
    else:
        # ── Viz 1 : Top 30 verbes globaux ──────────────────────────────────
        top_v = df_verbs['verbe'].value_counts().head(100)
        fig, ax = plt.subplots(figsize=(13, 6))
        top_v[::-1].plot(kind='barh', ax=ax,
                         color=sns.color_palette('mako', len(top_v))[::-1])
        ax.set_title('Top 100 – Verbes dans le contexte des objectName', fontsize=12)
        ax.set_xlabel('Fréquence')
        plt.tight_layout()
        out_v1 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_top100.png'
        plt.savefig(out_v1, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Sauvegardé : {out_v1}')

        # ── Viz 2 : Heatmap verbe × ref (identifiés seulement) ─────────────
        df_verbs_id = df_verbs[df_verbs['has_ref']]
        if not df_verbs_id.empty:
            TOP_R, TOP_V = 100, 100
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8, len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 2b : Heatmap verbe × objectName (tous, identifiés ou non) ─────────────
    
            TOP_R, TOP_V = 100, 100
            top_refs_v  = df_verbs_id['ref'].value_counts().head(TOP_R).index
            top_verbs_v = df_verbs_id['verbe'].value_counts().head(TOP_V).index
            sub = df_verbs_id[
                df_verbs_id['ref'].isin(top_refs_v) &
                df_verbs_id['verbe'].isin(top_verbs_v)
            ]
            if not sub.empty:
                piv = sub.pivot_table(
                    index='verbe', columns='ref',
                    values='fichier', aggfunc='count', fill_value=0
                )
                fig2, ax2 = plt.subplots(
                    figsize=(max(12, len(piv.columns) * 0.75),
                             max(8, len(piv) * 0.5))
                )
                sns.heatmap(piv, cmap='Blues', linewidths=0.3,
                            annot=True, fmt='d', ax=ax2,
                            cbar_kws={'label': 'Co-occurrences'})
                ax2.set_title(
                    f'Co-occurrences verbe × objectName identifié'
                    f' (top {TOP_R} ref, top {TOP_V} verbes)',
                    fontsize=11, pad=12
                )
                ax2.tick_params(axis='x', rotation=45, labelsize=8)
                plt.tight_layout()
                out_v2 = Path(VISUALIZATIONS_DIR) / 'objectname_verbes_heatmap_general.png'
                plt.savefig(out_v2, dpi=150, bbox_inches='tight')
                plt.show()
                print(f'✓ Sauvegardé : {out_v2}')

        # ── Viz 3 : Profil verbal des 5 objectName identifiés les + fréquents
        if not df_verbs_id.empty:
            top5 = df_verbs_id['ref'].value_counts().head(5).index.tolist()
            n    = len(top5)
            fig3, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
            if n == 1:
                axes = [axes]
            for ax_i, ref_id in zip(axes, top5):
                sub_ref = (df_verbs_id[df_verbs_id['ref'] == ref_id]['verbe']
                           .value_counts().head(10))
                sub_ref[::-1].plot(
                    kind='barh', ax=ax_i,
                    color=sns.color_palette('rocket', len(sub_ref))[::-1]
                )
                ax_i.set_title(ref_id[:28], fontsize=9)
                ax_i.set_xlabel('Fréq.')
                ax_i.tick_params(labelsize=8)
            fig3.suptitle(
                'Profil verbal des 5 objectName identifiés les plus fréquents',
                fontsize=11, y=1.02
            )
            plt.tight_layout()
            out_v3 = Path(VISUALIZATIONS_DIR) / 'objectname_profil_verbal.png'
            plt.savefig(out_v3, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'✓ Sauvegardé : {out_v3}')


✓ 3,639 occurrences verbales dans les contextes KWIC
  838 lemmes verbaux distincts
✓ Sauvegardé : humanstica_output\objectname_verbes_top100.png
✓ Sauvegardé : humanstica_output\objectname_verbes_heatmap.png
✓ Sauvegardé : humanstica_output\objectname_verbes_heatmap_general.png
✓ Sauvegardé : humanstica_output\objectname_profil_verbal.png
